In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, RepeatedKFold
import pickle
import torch
import pickle
import re
from collections.abc import Generator, Iterable
from dataclasses import dataclass
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from chemprop.data import MoleculeDatapoint, MoleculeDataset
from chemprop.featurizers import MultiHotAtomFeaturizer, MultiHotBondFeaturizer, SimpleMoleculeMolGraphFeaturizer
from rdkit import Chem
from rdkit.Chem.rdchem import HybridizationType
from scipy.constants import (
    Avogadro,  # 1/mol
    Boltzmann,  # in J/K
)
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from ml_enhance import QuantumFPFileLoader, parallelize

In [16]:
with open("all_features.pkl", "rb") as f:
    data = pickle.load(f)

In [21]:
pd.DataFrame(data["atoms"]).to_csv("atom_features.csv")
pd.DataFrame(data["bonds"]).to_csv("bond_features.csv")
pd.DataFrame(data["mols"]).to_csv("mol_features.csv")

In [30]:
mol_features = pd.read_csv("mol_features.csv")

mol_features = mol_features if mol_features.size > 0 else None
mol_features is None

True

In [60]:
target_df = pd.read_csv("../data/processed_dataset_rerun.csv")[["smiles", "id", "solubility"]]

In [61]:
target_df["id"] = target_df["id"].apply(round)
target_df.to_csv("target_df.csv")

In [62]:
def build_nested_cv_splits(
    target_df,
    outer_cv,
    inner_cv,
    id_col="id",
):
    ids = target_df[id_col].to_numpy()

    nested_splits = {}

    for outer_i, (outer_tr_idx, outer_te_idx) in enumerate(outer_cv.split(ids)):
        outer_train_ids = ids[outer_tr_idx]
        outer_test_ids = ids[outer_te_idx]

        inner_folds = []

        for inner_tr_idx, inner_val_idx in inner_cv.split(outer_train_ids):
            inner_train_ids = outer_train_ids[inner_tr_idx]
            inner_val_ids = outer_train_ids[inner_val_idx]

            inner_folds.append(
                {
                    "train_ids": inner_train_ids,
                    "val_ids": inner_val_ids,
                }
            )

        nested_splits[f"outer_fold_{outer_i}"] = {
            "train_ids": outer_train_ids,
            "test_ids": outer_test_ids,
            "inner_folds": inner_folds,
        }

    return nested_splits

In [63]:
outer_cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=15)
inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)

nested_splits = build_nested_cv_splits(target_df, outer_cv, inner_cv)

In [76]:
torch.save(nested_splits, "chemprop_splits.pt")

In [ ]:
pd.DataFrame(nested_splits).to_csv("chemprop_splits.csv")

,outer_fold_0,outer_fold_1,outer_fold_2,outer_fold_3,outer_fold_4,outer_fold_5,outer_fold_6,outer_fold_7,outer_fold_8,outer_fold_9,...,outer_fold_15,outer_fold_16,outer_fold_17,outer_fold_18,outer_fold_19,outer_fold_20,outer_fold_21,outer_fold_22,outer_fold_23,outer_fold_24
train_ids,"[0, 1, 10, 100, 1000, 1002, 1003, 1004, 1007, ...","[0, 1, 100, 1001, 1002, 1003, 1004, 1005, 1008...","[10, 1000, 1001, 1003, 1005, 1007, 1008, 1009,...","[0, 1, 10, 100, 1000, 1001, 1002, 1003, 1004, ...","[0, 1, 10, 100, 1000, 1001, 1002, 1004, 1005, ...","[1, 10, 1000, 1001, 1002, 1003, 1005, 1007, 10...","[0, 1, 10, 100, 1000, 1001, 1002, 1004, 1005, ...","[0, 1, 10, 100, 1002, 1003, 1004, 1005, 1007, ...","[0, 1, 100, 1000, 1001, 1002, 1003, 1004, 1005...","[0, 10, 100, 1000, 1001, 1003, 1004, 1007, 101...",...,"[0, 1, 10, 100, 1000, 1001, 1002, 1008, 1009, ...","[0, 1, 10, 100, 1001, 1002, 1003, 1004, 1005, ...","[10, 1000, 1003, 1004, 1005, 1007, 1008, 1009,...","[0, 1, 10, 100, 1000, 1001, 1002, 1003, 1004, ...","[0, 1, 100, 1000, 1001, 1002, 1003, 1004, 1005...","[0, 10, 100, 1001, 1002, 1005, 1007, 1008, 100...","[1, 10, 100, 1000, 1002, 1003, 1004, 1005, 100...","[0, 1, 10, 100, 1000, 1001, 1002, 1003, 1004, ...","[0, 1, 1000, 1001, 1002, 1003, 1004, 1005, 100...","[0, 1, 10, 100, 1000, 1001, 1003, 1004, 1005, ..."
test_ids,"[1001, 1005, 1008, 1012, 1017, 1018, 1022, 102...","[10, 1000, 1007, 1011, 1014, 1025, 1026, 1027,...","[0, 1, 100, 1002, 1004, 102, 1020, 1032, 1033,...","[1013, 1021, 1024, 1029, 1031, 1037, 1038, 103...","[1003, 1009, 1010, 1015, 1016, 1019, 103, 1035...","[0, 100, 1004, 1021, 1026, 1034, 1042, 1048, 1...","[1003, 1010, 1013, 1016, 1017, 1019, 1032, 103...","[1000, 1001, 1012, 1014, 1015, 1018, 1028, 102...","[10, 1007, 1011, 1020, 1022, 1025, 103, 1041, ...","[1, 1002, 1005, 1008, 1009, 102, 1023, 1024, 1...",...,"[1003, 1004, 1005, 1007, 1010, 1021, 1024, 102...","[1000, 1008, 1014, 1018, 102, 1022, 1025, 1039...","[0, 1, 100, 1001, 1002, 1016, 1017, 1019, 1020...","[1011, 1015, 1023, 1028, 1029, 1036, 1041, 105...","[10, 1009, 1012, 1013, 1032, 1043, 1045, 105, ...","[1, 1000, 1003, 1004, 1011, 1014, 1016, 1019, ...","[0, 1001, 1018, 1027, 103, 1032, 1033, 1044, 1...","[1005, 1010, 1015, 1022, 1024, 1031, 1034, 103...","[10, 100, 1017, 1025, 1026, 1037, 1046, 1051, ...","[1002, 1007, 1008, 1009, 1012, 1013, 102, 1021..."
inner_folds,"[{'train_ids': [0, 1, 10, 100, 1000, 1002, 100...","[{'train_ids': [0, 1, 100, 1001, 1002, 1003, 1...","[{'train_ids': [10, 1000, 1001, 1003, 1005, 10...","[{'train_ids': [0, 1, 10, 100, 1000, 1001, 100...","[{'train_ids': [0, 1, 10, 100, 1000, 1001, 100...","[{'train_ids': [1, 10, 1000, 1001, 1002, 1003,...","[{'train_ids': [0, 1, 10, 100, 1000, 1001, 100...","[{'train_ids': [0, 1, 10, 100, 1002, 1003, 100...","[{'train_ids': [0, 1, 100, 1000, 1001, 1002, 1...","[{'train_ids': [0, 10, 100, 1000, 1001, 1003, ...",...,"[{'train_ids': [0, 1, 10, 100, 1000, 1001, 100...","[{'train_ids': [0, 1, 10, 100, 1001, 1002, 100...","[{'train_ids': [10, 1000, 1003, 1004, 1005, 10...","[{'train_ids': [0, 1, 10, 100, 1000, 1001, 100...","[{'train_ids': [0, 1, 100, 1000, 1001, 1002, 1...","[{'train_ids': [0, 10, 100, 1001, 1002, 1005, ...","[{'train_ids': [1, 10, 100, 1000, 1002, 1003, ...","[{'train_ids': [0, 1, 10, 100, 1000, 1001, 100...","[{'train_ids': [0, 1, 1000, 1001, 1002, 1003, ...","[{'train_ids': [0, 1, 10, 100, 1000, 1001, 100..."


In [82]:
data = torch.load("chemprop_splits.pt", weights_only=False)

In [83]:
data

{'outer_fold_0': {'train_ids': array([  0,   1,  10, ..., 997, 998, 999], shape=(7007,)),
  'test_ids': array([1001, 1005, 1008, ...,  976,  978,  982], shape=(1752,)),
  'inner_folds': [{'train_ids': array([  0,   1,  10, ..., 995, 996, 999], shape=(5605,)),
    'val_ids': array([1007, 1015, 1016, ...,  991,  997,  998], shape=(1402,))},
   {'train_ids': array([  0,   1,  10, ..., 997, 998, 999], shape=(5605,)),
    'val_ids': array([1013,  103, 1033, ...,  992,  994,  995], shape=(1402,))},
   {'train_ids': array([  1,  10, 100, ..., 995, 997, 998], shape=(5606,)),
    'val_ids': array([   0, 1003, 1004, ...,  993,  996,  999], shape=(1401,))},
   {'train_ids': array([   0,  100, 1000, ...,  997,  998,  999], shape=(5606,)),
    'val_ids': array([   1,   10, 1010, ...,  983,  987,  989], shape=(1401,))},
   {'train_ids': array([  0,   1,  10, ..., 997, 998, 999], shape=(5606,)),
    'val_ids': array([ 100, 1000, 1002, ...,  984,  986,  990], shape=(1401,))}]},
 'outer_fold_1': {'trai

In [17]:
with open("chemprop_splits.pkl", "rb") as f:
    data = pickle.load(f)

data

{'outer_fold_0': {'train_ids': array([  0,   1,  10, ..., 997, 998, 999], shape=(7007,)),
  'test_ids': array([1001, 1005, 1008, ...,  976,  978,  982], shape=(1752,)),
  'inner_folds': [{'train_ids': array([  0,   1,  10, ..., 995, 996, 999], shape=(5605,)),
    'val_ids': array([1007, 1015, 1016, ...,  991,  997,  998], shape=(1402,))},
   {'train_ids': array([  0,   1,  10, ..., 997, 998, 999], shape=(5605,)),
    'val_ids': array([1013,  103, 1033, ...,  992,  994,  995], shape=(1402,))},
   {'train_ids': array([  1,  10, 100, ..., 995, 997, 998], shape=(5606,)),
    'val_ids': array([   0, 1003, 1004, ...,  993,  996,  999], shape=(1401,))},
   {'train_ids': array([   0,  100, 1000, ...,  997,  998,  999], shape=(5606,)),
    'val_ids': array([   1,   10, 1010, ...,  983,  987,  989], shape=(1401,))},
   {'train_ids': array([  0,   1,  10, ..., 997, 998, 999], shape=(5606,)),
    'val_ids': array([ 100, 1000, 1002, ...,  984,  986,  990], shape=(1401,))}]},
 'outer_fold_1': {'trai

In [88]:
nested_splits["outer_fold_0"]["test_ids"][0]

np.int64(1001)